# Maize data cleaning: field quality, labels and growing seasons

Author: **Pamely**. Submission folder: `notebook/Pamely/`.

## Rationale
I prefer conservative **pixel masking with explicit uncertainty** to deleting entire fields, zero-filling clouds, or removing high/zero yields automatically. SCL 4/5 preserves vegetation and bare soil (including possible crop failures); all ten bands and known offset metadata are required. A date must have at least 50% usable field pixels. Compare 70% and 90% as sensitivity checks; 50% is a practical starting policy, not a demonstrated optimum. For points this is effectively 0/100%, so threshold comparisons chiefly affect polygon fields. No model/test scores or yield values select this threshold.

Use measured planting-to-harvest dates when basic checks pass, otherwise clearly flag fixed seasonal estimates. Separate imagery quality from label eligibility: keep every maize field in the audit, including missing/zero yields and no retained imagery. Missing targets are never imputed. No blanket yield clipping or moisture correction is justified by these heterogeneous protocols.

This notebook covers **all maize country-years present in the supplied archive**: Kenya 2016–2020; Malawi 2020, 2022, 2023; Zimbabwe 2022–2024. Rwanda, Tanzania and Nigeria are not present in this archive and cannot be cleaned from it. Other Zimbabwe crops are outside the maize project scope. Unknown or unavailable sources are not silently counted as covered.

## Dataset access
[Download the preprocessed dataset from Google Drive](https://drive.google.com/file/d/1BqBGkxKgYLh2fat50dKRpOfeakgG6vPu/view?usp=sharing)

The archive is `maize_cleaned_submission.tar.gz`. The notebook creates this portable dataset bundle under the ignored `data/` directory. No full dataset belongs in Git.

The older `cropyield.duckdb` is NOT the required input. This pipeline requires `passarchive_v1.duckdb`, which includes acquisition-level pixels and SCL.


## 1. Environment and paths
Install `requirements.txt` beside this notebook into your Python 3.12 environment. Run from this folder or set `CLEANING_HOME` to it. Put the archive in the repository `data/` folder, or set `PASSARCHIVE_SOURCE` to its absolute path. Output directories are configurable and are never the source file. The first quality audit may take time and disk space; subsequent runs reuse it only after checking provenance.


In [1]:
from pathlib import Path
import os, sys, json, hashlib
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path(os.environ.get('CLEANING_HOME', Path.cwd())).resolve()
assert (HERE/'src/cropyield/satellite/passarchive_quality.py').is_file(), 'Run from this notebook folder or set CLEANING_HOME'
sys.path.insert(0, str(HERE/'src'))
sys.path.insert(0, str(HERE/'scripts'))
from cropyield.satellite.passarchive_quality import audit
from cropyield.satellite.passarchive_export import export_filtered
from export_passarchive_csv import convert
from export_passarchive_growing_seasons import export_seasons

SOURCE = Path(os.environ.get('PASSARCHIVE_SOURCE', HERE.parents[1]/'data/passarchive_v1.duckdb')).resolve()
WORK = Path(os.environ.get('CLEANING_WORK', HERE/'data')).resolve()
QUALITY = Path(os.environ.get('PASSARCHIVE_AUDIT', WORK/'quality_audit.duckdb')).resolve()
FILTERED = Path(os.environ.get('PASSARCHIVE_FILTERED', WORK/'filtered_50')).resolve()
SEASONS = FILTERED/'growing_seasons'
EXPORT = False  # True reproduces full pixel, CSV and growing-season exports (several GB).
assert SOURCE.is_file(), f'Missing source: {SOURCE}'
SOURCE_STAT = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
COUNTRIES = {'malawi':'MWI', 'zimbabwe':'ZWE', 'kenya':'KEN'}
def source_sql(query):
    with duckdb.connect(str(SOURCE), read_only=True) as c:
        return c.execute(query).df()
print('Python:', sys.executable)
print('Source:', SOURCE)


Python: /Users/speakeasy/Documents/workspace/DSC-291-Final-Project/.venv/bin/python
Source: /Users/speakeasy/Documents/workspace/DSC-291-Final-Project/data/passarchive_v1.duckdb


## 2. Inventory and label audit before filtering
Count independent field-seasons, not satellite rows. Preserve provider definitions, unit conversions, source identity and split membership. A missing yield is different from a recorded zero. Location-quality flags cannot be fixed by treating a rounded or missing coordinate as a precise field. Ambiguous source overlap is flagged for review rather than resolved using target values.


In [2]:
inventory = source_sql("""SELECT country_iso3,harvest_year,dataset,crop,count(*) AS field_seasons
FROM fields GROUP BY ALL ORDER BY country_iso3,harvest_year,dataset,crop""")
display(inventory)
labels = source_sql("""SELECT f.*,l.* EXCLUDE(fs_key,field_season_id,crop)
FROM fields f JOIN labels l USING(fs_key,field_season_id)
WHERE f.crop='maize'""")
assert not labels[['fs_key','field_season_id']].duplicated().any()
label_inventory = labels.groupby(['country_iso3','harvest_year','label_status'],dropna=False).size().rename('field_seasons').reset_index()
display(label_inventory)
display(labels.groupby(['country_iso3','location_quality'],dropna=False).size().rename('field_seasons').reset_index())


country_iso3  harvest_year                        dataset    crop  field_seasons
         KEN          2016          one_acre_fund_curated   maize             58
         KEN          2016             one_acre_fund_full   maize           3876
         KEN          2017          one_acre_fund_curated   maize             11
         KEN          2017             one_acre_fund_full   maize           3991
         KEN          2018          one_acre_fund_curated   maize             35
         KEN          2018             one_acre_fund_full   maize           4245
         KEN          2019          one_acre_fund_curated   maize             77
         KEN          2019             one_acre_fund_full   maize           5372
         KEN          2020          one_acre_fund_curated   maize            208
         KEN          2020             one_acre_fund_full   maize           6190
         MWI          2020             one_acre_fund_full   maize           2594
         MWI          2022  

## 3. Pixel quality and threshold sensitivity
The supplied helper validates unique field/product/footprint keys before joining. Required bands are B02/B03/B04/B08 at native 10 m and B05/B06/B07/B8A/B11/B12 at native 20 m. SCL remains native 20 m information: repeating it to 10 m support adds no detail. Usable pixels require SCL 4/5, complete positive native-band samples and known reflectance offsets. Negative reflectance after a documented offset is not automatically an invalid DN.

Cloud/cirrus (8/9/10), shadow (3), snow (11), invalid (0/1), water (6), uncertain (2/7) and unknown SCL do not count as usable. Coverage is usable pixels / the **full expected footprint**, so missing pixels cannot shrink the denominator. One product/tile is selected deterministically using L2A availability, footprint support, processing baseline and identifiers rather than yield or the least-cloudy product. The numerator is measured only within that selected product.


In [3]:
if not QUALITY.exists():
    audit(SOURCE, QUALITY).close()
with duckdb.connect(str(QUALITY), read_only=True) as c:
    provenance = c.execute('SELECT * FROM audit_provenance').fetchone()
    expected = (SOURCE.name, *SOURCE_STAT, hashlib.sha256((HERE/'src/cropyield/satellite/passarchive_quality.py').read_bytes()).hexdigest())
    assert provenance == expected, 'Audit source or algorithm changed; use a fresh output path'
    checks = c.execute('SELECT * FROM validation').df()
    assert checks.passed.all()
    retention = c.execute("SELECT * FROM retention_by_dataset WHERE crop='maize' ORDER BY country_iso3,harvest_year,dataset,threshold_percent").df()
    coverage = c.execute("SELECT * FROM retention_by_month WHERE crop='maize'").df()
display(checks)
display(retention)
# This comparison deliberately includes all available maize nations/years.
comparison = retention.groupby(['country_iso3','harvest_year','threshold_percent'])[['candidate_dates','retained_dates']].sum().reset_index()
comparison['retained_percent'] = 100*comparison.retained_dates/comparison.candidate_dates
display(comparison)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,4))
for country, group in comparison.groupby('country_iso3'):
    means = group.groupby('threshold_percent').retained_percent.mean()
    ax.plot(means.index, means.values, marker='o', label=country)
ax.set(xlabel='Minimum usable footprint (%)', ylabel='Mean country-year retention (%)', title='Sensitivity of acquisition retention (equal weight per year)')
ax.legend(); plt.show()


                      check_name  passed
   unique field acquisition keys    True
  usable counts within footprint    True
flag categories within footprint    True
          threshold monotonicity    True
 threshold_percent                        dataset country_iso3  crop  harvest_year  candidate_dates  fields_with_candidates  dates_without_footprint  retained_dates  fields_with_retained_dates  retained_percent
                50             one_acre_fund_full          KEN maize          2016           240859                    3770                        0           91492                        3770         37.985709
                70             one_acre_fund_full          KEN maize          2016           240859                    3770                        0           91492                        3770         37.985709
                90             one_acre_fund_full          KEN maize          2016           240859                    3770                        0           914

## 4. Export masked imagery and estimated growing seasons
At the 50% rule, retain the full expected pixel grid and mask unusable band values with NULL, never 0. Compute field spectral medians using usable pixels only. Apply `(DN + product offset) / 10000` once; exported bands are already reflectance. Do not rescale them again.

The full-window export preserves `previous` and `current` archive years. The season export uses recorded dates only if both exist, planting belongs to label year or preceding year, harvest belongs to label year or following year, harvest follows planting, and duration is at most 400 days. This last limit is a review rule, not a biological maturity claim. Otherwise use Oct 1–Jun 30 for Malawi/Zimbabwe and Feb 1–Sep 30 for Kenya, recording the fallback reason. Kenya's fallback may truncate later harvests. Missing months remain missing; no seasonal interpolation is performed.

These are **retrospective** sequences. Recorded harvest dates cannot define an operational forecast cutoff. Fixed windows can contain preplanting/postharvest scenes; incomplete archive windows are flagged. Train/test preprocessing, any interpolation and statistical outlier treatment must be fit inside training folds only.


In [4]:
if EXPORT:
    if not (FILTERED/'manifest.json').is_file():
        export_filtered(SOURCE, QUALITY, FILTERED, threshold=50)
    if not (FILTERED/'csv/manifest.json').is_file():
        convert(FILTERED, FILTERED/'csv')
    if not (SEASONS/'manifest.json').is_file():
        export_seasons(FILTERED, SEASONS)
assert (SEASONS/'manifest.json').is_file(), 'Set EXPORT=True to reproduce the export, or point PASSARCHIVE_FILTERED to an existing export'
parent_manifest = json.loads((FILTERED/'manifest.json').read_text())
season_manifest = json.loads((SEASONS/'manifest.json').read_text())
assert parent_manifest['complete'] and season_manifest['complete']
assert parent_manifest['source_bytes'] == SOURCE_STAT[0] and parent_manifest['source_mtime_ns'] == SOURCE_STAT[1]
assert parent_manifest['audit_algorithm_sha256'] == expected[-1]
assert parent_manifest['export_algorithm_sha256'] == hashlib.sha256((HERE/'src/cropyield/satellite/passarchive_export.py').read_bytes()).hexdigest()
assert season_manifest['script_sha256'] == hashlib.sha256((HERE/'scripts/export_passarchive_growing_seasons.py').read_bytes()).hexdigest()
assert season_manifest['parent_manifest_sha256'] == hashlib.sha256((FILTERED/'manifest.json').read_bytes()).hexdigest()
display(pd.read_csv(SEASONS/'summary.csv'))


 country  original_observations  seasonal_observations  original_field_seasons_with_imagery  seasonal_field_seasons_with_imagery  recorded_date_observations  fixed_window_observations                        file     bytes
  malawi                 291348                  51688                                 2628                                 2628                       21710                      29978   malawi_growing_season.csv  41783836
zimbabwe                 776489                 225946                                 7324                                 7324                       30250                     195696 zimbabwe_growing_season.csv 207641057
   kenya                1629770                 406199                                23158                                23157                      293329                     112870    kenya_growing_season.csv 323177962


## 5. Label eligibility and country-year checks
All labels are preserved in the exported field audit. The suggested supervised subset requires a finite nonnegative yield, a field-scale usable location with finite WGS84 coordinates, resolved source identity, and at least one retained seasonal observation. Recorded zeros remain valid targets. Values above 15 t/ha are marked for review rather than clipped or automatically deleted: a universal cap can remove real productive fields. Missing labels, missing imagery, ambiguous identity and unusable positions each receive explicit reasons.

This subset is only a starting input for modeling. Unknown moisture basis, intercropping, irrigation and protocol differences are retained as metadata; they do not support a universal physical correction. The notebook does not claim a pooled multicountry score. Preserve field identities, frozen splits and source-overlap groups across partitions; split versions must not be joined together as repeated examples.


In [5]:
from prepare_dataset import prepare
BUNDLE_DIR = WORK/'submission_dataset'
# Refuses to overwrite; choose a new folder to produce a fresh version.
if not (BUNDLE_DIR/'manifest.json').is_file():
    prepare(FILTERED, QUALITY, BUNDLE_DIR)
bundle_manifest = json.loads((BUNDLE_DIR/'manifest.json').read_text())
assert bundle_manifest['complete']
assert bundle_manifest['season_manifest_sha256'] == hashlib.sha256((SEASONS/'manifest.json').read_bytes()).hexdigest()
for item in bundle_manifest['files']:
    assert hashlib.sha256((BUNDLE_DIR/item['path']).read_bytes()).hexdigest()==item['sha256'], item['path']
summary = pd.read_csv(BUNDLE_DIR/'country_year_summary.csv')
display(summary)
expected_years = set(map(tuple, inventory.loc[inventory.crop.eq('maize'),['country_iso3','harvest_year']].drop_duplicates().to_numpy()))
assert set(map(tuple,summary[['country_iso3','harvest_year']].drop_duplicates().to_numpy()))==expected_years
assert (SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)==SOURCE_STAT
print('Verified all maize country-years; source unchanged.')


country_iso3  harvest_year  fields_total  fields_with_imagery  supervised_fields  high_yield_review  seasonal_observations
         KEN          2016          3934                 3770               3766                  0                  48970
         KEN          2017          4002                 3990               3984                  0                  73955
         KEN          2018          4280                 4230               4230                  0                  69823
         KEN          2019          5449                 5348               5339                  0                 100199
         KEN          2020          6398                 5819               5815                  0                 113252
         MWI          2020          2594                 1832               1738                  0                  25269
         MWI          2022           529                  529                529                  0                  17706
         MWI    

## 6. Load a manageable sample and share the dataset
The bundle contains three growing-season Parquet tables, one label/eligibility audit per country, season-coverage CSVs, unchanged split definitions/memberships, country-year summaries and file hashes. Only this code and aggregate notebook outputs go in Git. The preprocessed dataset is linked in the Dataset access section at the top of this notebook.

Use `supervised_eligible` from each field audit before training. The same yield repeats across acquisitions; build one sequence per field-season rather than scoring acquisitions as independent labels. Group and split before fitting imputers/scalers. The preview below loads only a few columns and rows and does not expose field coordinates in committed notebook outputs.


In [6]:
with duckdb.connect() as c:
    preview = c.execute("""SELECT field_season_id,harvest_year,sensing_time_utc,season_basis,
        usable_percent,b04,b08,yield_t_ha FROM read_parquet(?) LIMIT 5""",
        [str(BUNDLE_DIR/'malawi/observations.parquet')]).df()
print('Preview available in `preview`:',preview.shape)
# display(preview) locally if needed; raw records are intentionally not saved in Git outputs.
import tarfile
archive = WORK/'maize_cleaned_submission.tar.gz'
if not archive.exists():
    with tarfile.open(archive,'w:gz') as t:
        t.add(BUNDLE_DIR,arcname='maize_cleaned')
print('Upload this file to Drive:',archive)
print('Archive size (MB):',round(archive.stat().st_size/1e6,1))


Preview available in `preview`: (5, 8)
Upload this file to Drive: /Users/speakeasy/Documents/workspace/DSC291DSSD/notebook/Pamely/data/maize_cleaned_submission.tar.gz
Archive size (MB): 50.8


## Limits and next checks
The 50/70/90 comparison measures retention, not prediction accuracy. Evaluate crop-protocol compatibility and missingness by region before choosing a final policy. No claim is made that 50% is optimal, that all catalogued dates have observations, that unknown zeros mean failure, or that a fallback calendar is a verified biological season. This work compares reasonable cleaning alternatives without optimizing on held-out yields. Tests cover masking, missing bands, native grids, zeros, CSV null round trips and season boundaries: run `PYTHONPATH=src python -m pytest tests -q` from this notebook folder.
